In [2]:
# Library imports
import os
import pandas as pd
import datetime

# File paths and names mapping (for better readability in output)
file_mapping = {
    "adv_mt": {
        "path": "Advancer MT_ServiceWatch Log Parameters - 26th August 24.xlsx",
        "sheet": "AMT ServiceWatch Log Parameters",
        "display_name": "AMT"
    },
    "adv_st": {
        "path": "Advancer ST_REQ_5221_ServiceWatch Log Parameters - NAVI ST - 14th Oct 2024.xlsx",
        "sheet": "ServiceWatch Log Parameters",
        "display_name": "Navigator"
    },
    "APU": {
        "path": "APU TriPac_ServiceWatch Log Parameters_G3_PL5_09th June 25 .xlsx",
        "sheet": "APU ServiceWatch Log Params",
        "display_name": "APU"
    },
    "DEET_MT": {
        "path": "DEET_MT_ServiceWatch Version 1.5.xlsx",
        "sheet": "ServiceWatch Specification",
        "display_name": "DMT"
    },
    "DEET_ST": {
        "path": "DEET_ST_Synergy_SW_Parameters_Rev5.2 REQ 552-update.xlsx",
        "sheet": "Sheet2",
        "display_name": "DEET"
    },
    "Nebula": {
        "path": "Nebula_Galaxy_sw_requirements_v7.xlsx",
        "sheet": "ServiceWatch Log Parameters",
        "display_name": "Nebula-Galaxy"
    },
    "RB": {
        "path": "Railblazer_Service watch data logger- Advancer-sDRC - 29th September 23.xlsx",
        "sheet": "ServiceWatch Log Parameters",
        "display_name": "RB"
    }
}

# Dictionary to store SPN mappings
SPN_dict = {}

# Read each file and process SPNs
for key, info in file_mapping.items():
    try:
        # Read the Excel file
        df = pd.read_excel(info["path"], sheet_name=info["sheet"])
        print(f"\nColumns in {info['display_name']}:")
        print(df.columns.tolist())
        
        # Find the correct column name
        spn_column = None
        for col in ['SPN', 'Parameter ID', 'Parameter ID (new)']:
            if col in df.columns:
                spn_column = col
                break
        
        if spn_column is None:
            print(f"Warning: No SPN column found in {info['display_name']}")
            continue

        # Process SPNs
        for spn in df[spn_column].dropna():  # Skip NaN values
            if spn not in SPN_dict:
                SPN_dict[spn] = set()  # Use a set instead of list to prevent duplicates
            SPN_dict[spn].add(info["display_name"])
            
        # Create the output DataFrame
        result_df = pd.DataFrame({
            'SPN': list(SPN_dict.keys()),
            'Platforms': [', '.join(sorted(platforms)) for platforms in SPN_dict.values()],  # Convert set to sorted list
            'Count': [len(platforms) for platforms in SPN_dict.values()]
        })
            
    except Exception as e:
        print(f"Error processing {info['display_name']}: {str(e)}")

# Create the output DataFrame
result_df = pd.DataFrame({
    'SPN': list(SPN_dict.keys()),
    'Platforms': [', '.join(platforms) for platforms in SPN_dict.values()],
    'Count': [len(platforms) for platforms in SPN_dict.values()]
})

# Sort by count (descending) and SPN
result_df = result_df.sort_values(['Count', 'SPN'], ascending=[False, True])


# Create Output directory if it doesn't exist
output_dir = "Outputs"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Generate filename with timestamp to avoid overwrite issues
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_filename = os.path.join(output_dir, f"SPN_Common_{timestamp}.csv")

# Save to Excel
result_df.to_csv(output_filename, index=False)
# result_df.to_excel(output_filename, index=False)
print(f"\nResults saved to {output_filename}")
print(f"Found {len(SPN_dict)} unique SPNs across {len(file_mapping)} platforms")


Columns in AMT:
['SPN', 'Parameter Name (as read in SW data log)', 'Display Type(s)', 'Log Type', 'TIME ?', 'Event?', 'Dependency', '(WinTrac)\nGraph', '(WinTrac) Access Level', 'Event log when alarm code(s) set', 'Auto Log at Power ON? ', 'Auto Log at 12:05? ', 'Event Log When Parameter Changes? ', 'Additional Reqs', 'Documentation Notes/Comments']

Columns in Navigator:
['ID', 'Acronym (old)', 'Acronym (new)', 'Parameter ID (new)', 'Parameter ID (old)', 'ST Name', 'Display Types', 'Default Log', 'TIME ?', 'Event?', 'Dependency', 'Access Level', 'Event log when alarm code(s) set', 'Additional Reqs', 'Documentation Notes']

Columns in APU:
['Acronym (old)', 'Acronym (new)', 'Parameter ID (new)', 'Parameter ID (old)', 'ST Name', 'Display Types', 'Default Log', 'TIME ?', 'Event?', 'Dependency', '(WinTrac)\nGraph', 'Access Level', 'Event log when alarm code(s) set', 'Auto Log at Power ON?', 'Auto Log at 12:05?', 'Event Log When Parameter Changes?', 'Additional Reqs', 'Documentation Notes

In [8]:
def verify_spn_data(platform_key="APU"):
    # Read original file
    info = file_mapping[platform_key]
    original_df = pd.read_excel(info["path"], sheet_name=info["sheet"])
    
    # Find SPN column
    spn_column = None
    for col in ['SPN', 'Parameter ID', 'Parameter ID (new)']:
        if col in original_df.columns:
            spn_column = col
            break
    
    if spn_column is None:
        print(f"Error: No SPN column found in {info['display_name']}")
        return
    
    # Get original SPNs with more detailed analysis
    all_spns = original_df[spn_column]
    total_rows = len(all_spns)
    null_spns = all_spns.isna().sum()
    duplicate_spns = all_spns.duplicated().sum()
    unique_spns = len(all_spns.dropna().unique())
    
    print(f"\nDetailed Analysis for {info['display_name']}:")
    print(f"Total rows in SPN column: {total_rows}")
    print(f"Null/Empty SPNs: {null_spns}")
    print(f"Duplicate SPNs: {duplicate_spns}")
    print(f"Unique SPNs (excluding nulls): {unique_spns}")
    
    # Original verification code continues...
    original_spns = set(original_df[spn_column].dropna())
    result_spns = set()
    for idx, row in result_df.iterrows():
        if info["display_name"] in row["Platforms"]:
            result_spns.add(row["SPN"])
    
    # Print specific values for investigation
    print("\nFirst 5 SPNs from original file:")
    print(all_spns.head().tolist())
    
    # Compare sets
    missing_spns = original_spns - result_spns
    extra_spns = result_spns - original_spns
    
    if missing_spns or extra_spns:
        if missing_spns:
            print(f"\nMissing SPNs ({len(missing_spns)}):")
            for spn in sorted(missing_spns):
                print(f"- {spn}")
                # Show row details for missing SPNs
                rows = original_df[original_df[spn_column] == spn]
                print(f"  Row details: {rows.to_dict('records')}")
    else:
        print("\nAll SPNs matched perfectly! ✓")

# Run verification

verify_spn_data("adv_mt")
verify_spn_data("adv_st")
verify_spn_data("DEET_MT")
verify_spn_data("DEET_ST")
verify_spn_data("Nebula")
verify_spn_data("RB")
verify_spn_data("APU")  # Example call to verify SPN data for A


Detailed Analysis for Advancer MT:
Total rows in SPN column: 174
Null/Empty SPNs: 0
Duplicate SPNs: 0
Unique SPNs (excluding nulls): 174

First 5 SPNs from original file:
['0x0109', '0x010A', '0x010F', '0x010C', '0x010D']

All SPNs matched perfectly! ✓

Detailed Analysis for Advancer ST:
Total rows in SPN column: 107
Null/Empty SPNs: 0
Duplicate SPNs: 2
Unique SPNs (excluding nulls): 105

First 5 SPNs from original file:
['????', '0x04D2', '0x048F', '0x0098', '0x010F']

All SPNs matched perfectly! ✓

Detailed Analysis for DEET MT:
Total rows in SPN column: 223
Null/Empty SPNs: 0
Duplicate SPNs: 5
Unique SPNs (excluding nulls): 218

First 5 SPNs from original file:
['0x0209', '0x0098', '0x0852', '0x0851', '0x0853']

All SPNs matched perfectly! ✓

Detailed Analysis for DEET ST:
Total rows in SPN column: 512
Null/Empty SPNs: 106
Duplicate SPNs: 109
Unique SPNs (excluding nulls): 402

First 5 SPNs from original file:
['????', '????', '0x002D', '0x004A', '0x0058']

All SPNs matched perfect